# **Capstone Project: A Bi-Objective Evolutionary Approach to Feature Selection for Customer Value Prediction in Fintech**

# *Initial Exploratory Data Analysis*

## MBAI 5600G: Applied Integrative Analytics Capstone Project

### Group 7: Brennan Mason & Mohammad Shah
---

## Environment Setup

In [ ]:
# Specify base path to local directory
BASE_PATH = "/content/drive/Shareddrives/MBAI Capstone S S26 Group 7/"

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Ingestion & Inspection

In [ ]:
import pandas as pd

# Define path
path = BASE_PATH + "p2p-customer-value-prediction/data/interim/sampled_loans.csv"

# Load data
loans = pd.read_csv(path)

# Inspect head
loans.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,spread,issue_yr_mnth,avg_spread,min_spread,return,beta,disc_rate,loan_duration,avg_monthly_cash_flow,rar
0,49763530,NaN,7000.0,7000.0,7000.0,36 months,0.0789,219.00,A,A5,...,0.077472,2015-05,0.124520,0.051806,0.124084,0.404783,0.006530,32.0,245.893361,7126.312861
1,1614412,NaN,5000.0,5000.0,5000.0,36 months,0.0603,152.18,A,A1,...,0.058818,2012-10,0.137318,0.058518,0.091950,0.404783,0.007239,29.0,188.267241,4944.316301
2,19435976,NaN,10000.0,10000.0,10000.0,36 months,0.0712,309.32,A,A3,...,0.070246,2014-06,0.138971,0.059056,0.113552,0.404783,0.007315,36.0,309.319915,9830.124029
3,44756872,NaN,12000.0,12000.0,12000.0,36 months,0.0593,364.69,A,A1,...,0.057898,2015-04,0.124802,0.057898,0.093717,0.404783,0.006820,36.0,364.572381,11681.851552
4,108855699,NaN,6700.0,6700.0,6700.0,36 months,0.0532,201.77,A,A1,...,0.042755,2017-05,0.126481,0.042755,0.029875,0.404783,0.006173,8.0,862.520378,6753.795702


In [ ]:
# Check size and type of data
loans.info(verbose=True, memory_usage="deep", show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268615 entries, 0 to 268614
Data columns (total 163 columns):
 #    Column                                      Non-Null Count   Dtype  
---   ------                                      --------------   -----  
 0    id                                          268615 non-null  int64  
 1    member_id                                   0 non-null       float64
 2    loan_amnt                                   268615 non-null  float64
 3    funded_amnt                                 268615 non-null  float64
 4    funded_amnt_inv                             268615 non-null  float64
 5    term                                        268615 non-null  object 
 6    int_rate                                    268615 non-null  float64
 7    installment                                 268615 non-null  float64
 8    grade                                       268615 non-null  object 
 9    sub_grade                                   268615 non-nu

In [ ]:
# Specify and drop intermediate features used to derive the target variable
cols = [
    "issue_yr",
    "avg_cdi_rate",
    "spread",
    "issue_yr_mnth",
    "avg_spread",
    "min_spread",
    "return",
    "beta",
    "disc_rate",
    "loan_duration",
    "avg_monthly_cash_flow"
]

loans.drop(cols, axis=1, inplace=True)
loans.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268615 entries, 0 to 268614
Columns: 152 entries, id to rar
dtypes: float64(115), int64(1), object(36)
memory usage: 696.0 MB


In [ ]:
from pandas.tseries.offsets import MonthEnd

# Convert issue_d to datetime type
loans["issue_d"] = pd.to_datetime(loans["issue_d"])

# Get date range
start_date = loans["issue_d"].min()
end_date = loans["issue_d"].max() + MonthEnd(0)

# Compute length of reference period in months
ref_period = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month) + 1

# Print results
print("Start Date:", start_date.strftime("%Y-%m-%d"))
print("End Date:", end_date.strftime("%Y-%m-%d"))
print(f"Reference Period Length: {ref_period} months")

Start Date: 2007-07-01
End Date: 2018-12-31
Reference Period Length: 138 months


In [ ]:
# Check for duplicate observations
loans.duplicated().sum()

np.int64(0)

## Missing Value Analysis

In [ ]:
# Define function to identify columns with missingness above a specified threshold
def check_missingness(df, threshold):
  """
  Identifies and returns sparse columns in a provided DataFrame with missingness above a specified threshold.

  Parameters:
  -----------
      - df (pandas.core.frame.DataFrame): DataFrame to analyze.
      - threshold (float): Proportion of missing values above which a column is considered sparse.

  Returns:
  --------
      - dict: Dictionary containing column names as keys and corresponding missingness percentages as values.
  """
  # Get total observation count
  n_obs = len(df)

  # Initialize empty dict to store sparse column names and respective missingness %s
  sparse_cols = {}

  # Iterate through columns
  for col in df.columns:
    # Compute missingness %
    pct_missing = df[col].isna().sum() / n_obs

    # Check if missingness % is above threshold
    if pct_missing > threshold:
      sparse_cols[col] = pct_missing

  # Sort dict desc
  sparse_cols = dict(sorted(sparse_cols.items(), key=lambda x: x[1], reverse=True))

  # Return dict
  return sparse_cols

# Identify and print columns with missingness above 50%
sparse_cols = check_missingness(loans, 0.5)

for i, (col, pct) in enumerate(sparse_cols.items(), start=1):
  print(f"{i}. {col}: {pct:.2%}")

1. member_id: 100.00%
2. next_pymnt_d: 100.00%
3. orig_projected_additional_accrued_interest: 99.73%
4. hardship_type: 99.58%
5. hardship_reason: 99.58%
6. hardship_status: 99.58%
7. deferral_term: 99.58%
8. hardship_amount: 99.58%
9. hardship_start_date: 99.58%
10. hardship_end_date: 99.58%
11. payment_plan_start_date: 99.58%
12. hardship_length: 99.58%
13. hardship_dpd: 99.58%
14. hardship_loan_status: 99.58%
15. hardship_payoff_balance_amount: 99.58%
16. hardship_last_payment_amount: 99.58%
17. sec_app_mths_since_last_major_derog: 99.52%
18. sec_app_revol_util: 98.65%
19. revol_bal_joint: 98.63%
20. sec_app_fico_range_low: 98.63%
21. sec_app_fico_range_high: 98.63%
22. sec_app_earliest_cr_line: 98.63%
23. sec_app_inq_last_6mths: 98.63%
24. sec_app_mort_acc: 98.63%
25. sec_app_open_acc: 98.63%
26. sec_app_open_act_il: 98.63%
27. sec_app_num_rev_accts: 98.63%
28. sec_app_chargeoff_within_12_mths: 98.63%
29. sec_app_collections_12_mths_ex_med: 98.63%
30. verification_status_joint: 98.1

In [ ]:
# Identify and print columns with any missing values
missing_cols = check_missingness(loans, 0.0)

for i, (col, pct) in enumerate(missing_cols.items(), start=1):
  print(f"{i}. {col}: {pct:.2%}")

1. member_id: 100.00%
2. next_pymnt_d: 100.00%
3. orig_projected_additional_accrued_interest: 99.73%
4. hardship_type: 99.58%
5. hardship_reason: 99.58%
6. hardship_status: 99.58%
7. deferral_term: 99.58%
8. hardship_amount: 99.58%
9. hardship_start_date: 99.58%
10. hardship_end_date: 99.58%
11. payment_plan_start_date: 99.58%
12. hardship_length: 99.58%
13. hardship_dpd: 99.58%
14. hardship_loan_status: 99.58%
15. hardship_payoff_balance_amount: 99.58%
16. hardship_last_payment_amount: 99.58%
17. sec_app_mths_since_last_major_derog: 99.52%
18. sec_app_revol_util: 98.65%
19. revol_bal_joint: 98.63%
20. sec_app_fico_range_low: 98.63%
21. sec_app_fico_range_high: 98.63%
22. sec_app_earliest_cr_line: 98.63%
23. sec_app_inq_last_6mths: 98.63%
24. sec_app_mort_acc: 98.63%
25. sec_app_open_acc: 98.63%
26. sec_app_open_act_il: 98.63%
27. sec_app_num_rev_accts: 98.63%
28. sec_app_chargeoff_within_12_mths: 98.63%
29. sec_app_collections_12_mths_ex_med: 98.63%
30. verification_status_joint: 98.1

In [ ]:
# Construct binned missingness summary table
missing_df = loans.isna().sum().reset_index()
missing_df.rename(columns={"index": "variable", 0: "missing_count"}, inplace=True)
missing_df["missing_pct"] = missing_df["missing_count"] / len(loans)

missing_df["missing_bin"] = pd.cut(
    missing_df["missing_pct"],
    bins=[0, (1 / len(loans)), 0.1301, 1.01],
    right=False,
    include_lowest=True,
    labels=["None", "Low", "High"]
)

missing_summary = missing_df.value_counts("missing_bin").sort_index(ascending=False).reset_index()
missing_summary.rename(columns={0: "variable_count"}, inplace=True)

missing_summary

,missing_bin,count
0,High,58
1,Low,45
2,None,49


## Distribution Analysis

In [ ]:
import plotly.graph_objects as go

# Visualize RAR distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=loans["rar"],
    nbinsx=40,
    marker=dict(color="steelblue"),
    name="RAR"
))

fig.update_layout(
    title="Distribution of Risk-Adjusted Revenue",
    xaxis_title="Risk-Adjusted Revenue ($)",
    yaxis_title="Frequency",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Get list of numeric columns
num_cols = loans.select_dtypes(include="number").columns.tolist()

In [ ]:
# Define function to identify numeric columns with skewness above a specified threshold
def check_skewness(df, cols, threshold=0.5):
  """
  Identifies and returns skewed numeric columns in a provided DataFrame with skewness above a specified threshold.

  Parameters:
  -----------
      - df (pandas.core.frame.DataFrame): DataFrame to analyze.
      - cols (list): List of numeric column names to analyze.
      - threshold (float): Absolute skewness value above which a column is considered skewed. Default is 0.5.

  Returns:
  --------
      - dict: Dictionary containing column names as keys and corresponding skewness values as values.
  """
  # Get subset of relevant (i.e., numeric) columns
  df = df[cols]

  # Initialize empty dict to store skewed column names and respective skewness
  skewed_cols = {}

  # Iterate through columns
  for col in df.columns:
    # Compute skewness
    skew = df[col].skew()

    # Check if absolute skewness is above threshold
    if abs(skew) > threshold:
      skewed_cols[col] = skew

  # Sort dict desc
  skewed_cols = dict(sorted(skewed_cols.items(), key=lambda x: abs(x[1]), reverse=True))

  # Return dict
  return skewed_cols

# Identify and print columns with skewed distributions
skewed_cols = check_skewness(loans, num_cols, threshold=0.5)

for i, (col, skew) in enumerate(skewed_cols.items(), start=1):
  print(f"{i}. {col}: {skew:.2f}")

1. tot_coll_amt: 147.17
2. delinq_amnt: 83.75
3. tax_liens: 51.50
4. num_tl_120dpd_2m: 38.08
5. dti: 31.11
6. annual_inc: 28.94
7. max_bal_bc: 26.16
8. sec_app_chargeoff_within_12_mths: 24.14
9. collections_12_mths_ex_med: 21.99
10. chargeoff_within_12_mths: 20.08
11. num_tl_30dpd: 19.32
12. pub_rec: 18.69
13. total_rec_late_fee: 16.87
14. acc_now_delinq: 15.95
15. num_tl_90g_dpd_24m: 13.37
16. revol_bal: 10.42
17. collection_recovery_fee: 8.87
18. sec_app_collections_12_mths_ex_med: 8.76
19. recoveries: 8.21
20. total_rev_hi_lim: 6.50
21. delinq_2yrs: 5.63
22. num_accts_ever_120_pd: 5.00
23. mo_sin_rcnt_tl: 4.48
24. avg_cur_bal: 3.88
25. total_bal_il: 3.84
26. bc_open_to_buy: 3.81
27. total_bal_ex_mort: 3.68
28. mths_since_rcnt_il: 3.53
29. mths_since_recent_bc: 3.47
30. mo_sin_rcnt_rev_tl_op: 3.44
31. pub_rec_bankruptcies: 3.38
32. total_bc_limit: 3.32
33. total_cu_tl: 3.31
34. total_il_high_credit_limit: 3.29
35. last_fico_range_low: -3.24
36. tot_hi_cred_lim: 3.19
37. sec_app_open_

In [ ]:
loans["tot_coll_amt"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.80, 0.85, 0.90, 0.95])

,tot_coll_amt
count,254957.000000
mean,236.317465
std,2659.740621
min,0.000000
5%,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
80%,0.000000
85%,50.000000


## Outlier Analysis

In [ ]:
# Define function to identify numeric columns with outliers
def id_outliers(df, cols, iqr_multiplier=1.5, z_threshold=3.0):
  """
  Identifies and returns numeric columns in a provided DataFrame containing outliers.
  Outliers are identified using either the IQR method or the Z-score method based on skewness.

  Parameters:
  -----------
      - df (pandas.core.frame.DataFrame): DataFrame to analyze.
      - cols (list): List of numeric column names to analyze.
      - iqr_multiplier (float): Multiplier for the IQR method. Default is 1.5.
      - z_threshold (float): Threshold for the Z-score method. Default is 3.0.

  Returns:
  --------
      - dict: Dictionary containing column names as keys and corresponding outlier counts as values.
  """
  # Get subset of relevant (i.e., numeric) columns
  df = df[cols]

  # Initialize empty dict to store names of columns with outliers and respective outlier counts
  outlier_cols = {}

  # Iterate through columns
  for col in df.columns:
    # Compute skewness
    skew = df[col].skew()

    # Assess skewness to determine appropriate outlier detection method
    if abs(skew) > 0.5:
      # Use IQR method
      # Compute Q1 and Q3
      q1 = df[col].quantile(0.25)
      q3 = df[col].quantile(0.75)

      # Compute IQR
      iqr = q3 - q1

      # Compute upper and lower fences
      uf = q3 + (iqr_multiplier * iqr)
      lf = q1 - (iqr_multiplier * iqr)

      # Detect outliers
      outlier_mask = (df[col] > uf) | (df[col] < lf)

    else:
      # Use Z-score method
      # Compute Z-scores (i.e., standardize observations)
      z_scores = (df[col] - df[col].mean()) / df[col].std()

      # Detect outliers
      outlier_mask = z_scores.abs() > z_threshold

    # Count outliers
    n_outliers = outlier_mask.sum()

    # Check if any outliers
    if n_outliers > 0:
      outlier_cols[col] = n_outliers

  # Sort dict desc
  outlier_cols = dict(sorted(outlier_cols.items(), key=lambda x: x[1], reverse=True))

  # Return dict
  return outlier_cols

# Identify and print columns containing outliers
outlier_cols = id_outliers(loans, num_cols, iqr_multiplier=1.5, z_threshold=3.0)

for i, (col, n_outliers) in enumerate(outlier_cols.items(), start=1):
  print(f"{i}. {col}: {n_outliers}")

1. num_accts_ever_120_pd: 60338
2. delinq_2yrs: 51635
3. pub_rec: 45582
4. tot_coll_amt: 39017
5. recoveries: 36869
6. collection_recovery_fee: 35140
7. pub_rec_bankruptcies: 33647
8. mths_since_recent_bc: 23983
9. bc_open_to_buy: 22369
10. total_rec_int: 19646
11. mo_sin_rcnt_rev_tl_op: 19547
12. pct_tl_nvr_dlq: 16460
13. revol_bal: 15951
14. total_bc_limit: 15685
15. num_il_tl: 15499
16. total_bal_ex_mort: 15314
17. mo_sin_rcnt_tl: 15310
18. total_rev_hi_lim: 14807
19. last_pymnt_amnt: 14687
20. num_tl_90g_dpd_24m: 14506
21. inq_last_6mths: 14383
22. open_il_24m: 13708
23. avg_cur_bal: 13573
24. annual_inc: 13450
25. total_il_high_credit_limit: 12964
26. total_rec_late_fee: 12028
27. num_bc_sats: 11507
28. num_op_rev_tl: 10698
29. fico_range_low: 9197
30. fico_range_high: 9197
31. open_acc: 9171
32. mths_since_rcnt_il: 9037
33. num_sats: 8940
34. total_cu_tl: 8910
35. open_act_il: 8820
36. tot_cur_bal: 8786
37. tot_hi_cred_lim: 8734
38. tax_liens: 8692
39. installment: 8379
40. mo_si

In [ ]:
import plotly.express as px

# Visualize Number of Accounts Ever 120+ Days Past Due distribution
fig = px.histogram(
    loans,
    x="num_accts_ever_120_pd",
    log_y=True, # This is the magic parameter
    title="Distribution of Number of Accounts Ever 120+ Days Past Due",
    template="seaborn",
    color_discrete_sequence=["steelblue"]
)

fig.update_layout(
    font=dict(family="Helvetica Neue, sans-serif", color="#666666"),
    xaxis_title="Number of Accounts",
    yaxis_title="Count of Borrowers (Log Scale)",
    width=1000,
    height=600
)

fig.update_yaxes(ticklabelstandoff=5, dtick=1)
fig.update_xaxes(ticklabelstandoff=5)

fig.show()

In [ ]:
loans["num_accts_ever_120_pd"].describe(percentiles=[0.25, 0.50, 0.75, 0.8, 0.85, 0.9, 0.95])

,num_accts_ever_120_pd
count,254957.000000
mean,0.506050
std,1.313531
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
80%,1.000000
85%,1.000000
90%,2.000000


In [ ]:
# Get top 10 features most correlated with target
corr_matrix = loans.corr(numeric_only=True)
corr_cols = corr_matrix.drop(index=["rar"]).abs().nlargest(10, "rar")["rar"].index.to_list()
corr_cols

['total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'funded_amnt',
 'loan_amnt',
 'funded_amnt_inv',
 'installment',
 'last_pymnt_amnt',
 'total_rec_int',
 'settlement_amount']

In [ ]:
from plotly.subplots import make_subplots

# Select top 6 nonredundant features
top_corr_cols = ["total_pymnt", "funded_amnt", "installment", "last_pymnt_amnt", "total_rec_int", "settlement_amount"]

# Construct boxplots to visualize feature distributions and outliers
fig = make_subplots(rows=2, cols=3, subplot_titles=top_corr_cols)

for i, col_name in enumerate(top_corr_cols):
  row = (i // 3) + 1
  col = (i % 3) + 1

  trace = go.Box(
      y=loans[col_name],
      marker_color="steelblue"
  )

  fig.add_trace(trace, row, col)

fig.update_layout(
    title="Distributions of Top 6 Features by Target Correlation",
    template="seaborn",
    showlegend=False,
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(showticklabels=False)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

## Variable Descriptions

In [ ]:
# Define path
path = BASE_PATH + "p2p-customer-value-prediction/references/data_dictionary.csv"

# Load data dictionary
data_dict = pd.read_csv(path)

# Strip whitespace from variable names and descriptions
data_dict["LoanStatNew"] = data_dict["LoanStatNew"].str.strip()
data_dict["Description"] = data_dict["Description"].str.strip()

for col in loans.columns:
  # Get variable description
  try:
    desc = data_dict.loc[data_dict["LoanStatNew"] == col, "Description"].values[0]
  except IndexError:
    desc = "N/A"

  # Append period if required
  if desc[-1] not in [".", "*"]:
    desc += "."

  # Get data type
  dtype = loans[col].dtype

  # Print clean list of variable names, types, and descriptions with markdown-friendly formatting
  print(f"* **{col} (*{dtype}*):** {desc}")

* **id (*int64*):** A unique LC assigned ID for the loan listing.
* **member_id (*float64*):** A unique LC assigned Id for the borrower member.
* **loan_amnt (*float64*):** The listed amount of the loan applied for by the borrower. If at some point in time, the credit department reduces the loan amount, then it will be reflected in this value.
* **funded_amnt (*float64*):** The total amount committed to that loan at that point in time.
* **funded_amnt_inv (*float64*):** The total amount committed by investors for that loan at that point in time.
* **term (*object*):** The number of payments on the loan. Values are in months and can be either 36 or 60.
* **int_rate (*float64*):** Interest Rate on the loan.
* **installment (*float64*):** The monthly payment owed by the borrower if the loan originates.
* **grade (*object*):** LC assigned loan grade.
* **sub_grade (*object*):** LC assigned loan subgrade.
* **emp_title (*object*):** The job title supplied by the Borrower when applying for t

## Categorical Exploration

In [ ]:
loans["emp_title"].value_counts().nlargest(50)

,count
emp_title,
Teacher,4237
Manager,3998
Owner,2099
Registered Nurse,1727
RN,1705
Supervisor,1676
Driver,1508
Sales,1504
Project Manager,1276


In [ ]:
cat_cols = loans.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
  print(loans[col].value_counts().nlargest(50))
  print()

term
36 months    203821
60 months     64794
Name: count, dtype: int64

grade
B    78471
C    76219
A    46988
D    40091
E    18650
F     6385
G     1811
Name: count, dtype: int64

sub_grade
C1    17001
B4    16717
B5    16396
B3    16317
C2    15747
C3    15066
B2    14941
C4    14853
B1    14100
C5    13552
A5    12747
A4    10496
D1    10335
D2     8967
A1     8594
D3     7778
A3     7663
A2     7488
D4     7019
D5     5992
E1     4796
E2     4209
E3     3634
E4     3107
E5     2904
F1     1982
F2     1470
F3     1180
F4      984
F5      769
G1      599
G2      431
G3      317
G4      239
G5      225
Name: count, dtype: int64

emp_title
Teacher                     4237
Manager                     3998
Owner                       2099
Registered Nurse            1727
RN                          1705
Supervisor                  1676
Driver                      1508
Sales                       1504
Project Manager             1276
Office Manager              1103
General Manager      

## Descriptive Statistics

In [ ]:
key_vars = ["funded_amnt", "int_rate", "installment", "annual_inc", "rar"]

loans[key_vars].describe().round(2)

In [ ]:
# Descriptive statistics for RAR target variable
print("RAR Descriptive Statistics:")
print(loans["rar"].describe().round(2))
print(f"Skewness: {loans['rar'].skew().round(4)}")
print(f"Kurtosis: {loans['rar'].kurt().round(4)}")

In [ ]:
# Identify highly skewed numeric features
numeric_cols = loans.select_dtypes(include="number").columns
skewness = loans[numeric_cols].skew().sort_values(ascending=False)
print("Top 10 Most Skewed Features:")
print(skewness.head(10))
print("\nFeatures with |skew| > 1:")
print(skewness[skewness.abs() > 1])



# Visualize top 10 most skewed numeric features
top10_skew = skewness.abs().sort_values(ascending=False).head(10).sort_values()

plt.figure(figsize=(9, 6))
top10_skew.plot(kind="barh", color="steelblue")
plt.axvline(x=1, color="red", linestyle="--", linewidth=1, label="Threshold (|skew| = 1)")
plt.title("Top 10 Most Skewed Numeric Features")
plt.xlabel("Absolute Skewness")
plt.ylabel("Feature")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Total features with |skew| > 1: {skewness[skewness.abs() > 1].shape[0]}")

## Correlation Analysis

In [ ]:
# Correlation matrix
# Compute correlation of all numeric features with RAR
numeric_cols = loans.select_dtypes(include="number").columns
rar_corr = loans[numeric_cols].corr()["rar"].drop("rar").sort_values(ascending=False)

print("Top 10 Positively Correlated Features with RAR:")
print(rar_corr.head(10).round(4))

print("\nTop 10 Negatively Correlated Features with RAR:")
print(rar_corr.tail(10).round(4))

In [ ]:
# Heatmap
# Focus heatmap on top 10 features correlated with RAR
top_10 = rar_corr.head(10).index.tolist()
top_10.append("rar")

corr_matrix_focused = loans[top_10].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix_focused,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Heatmap — Top 10 Features vs RAR")
plt.tight_layout()
plt.show()

In [ ]:
# Flagging high multicollinearity between features
# Identify feature pairs with correlation above 0.85
high_corr_pairs = []
corr_matrix_full = loans[numeric_cols].corr().abs()

for i in range(len(corr_matrix_full.columns)):
    for j in range(i+1, len(corr_matrix_full.columns)):
        if corr_matrix_full.iloc[i, j] > 0.85:
            high_corr_pairs.append({
                "Feature 1": corr_matrix_full.columns[i],
                "Feature 2": corr_matrix_full.columns[j],
                "Correlation": corr_matrix_full.iloc[i, j].round(4)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values("Correlation", ascending=False)
print(f"Number of highly correlated feature pairs (|r| > 0.85): {len(high_corr_df)}")
print(high_corr_df.head(40))